# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `pull_request_events` Silver Table

This step creates the Delta table used to store cleaned and structured **PullRequestEvent** records in the Silver layer.

The table contains three groups of fields:

- **Common event fields** — General GitHub event information such as the event ID, event timestamp, actor, and repository.
- **Bronze lineage fields** — Metadata retained from the Bronze layer so each Silver record can be traced back to its original source file and ingestion.
- **PullRequestEvent-specific fields** — Pull request lifecycle and change information, including the PR action, PR number, creation and closing timestamps, merge status, commit count, additions, deletions, and changed files.

These fields allow the Silver layer to represent important pull request behavior such as creation, closure, merging, development activity, and code-change size.

The table is stored using **Delta Lake** (`USING DELTA`) for reliable and structured storage.

`CREATE TABLE IF NOT EXISTS` makes the operation safe to rerun because an existing table will not be recreated or overwritten.

> **Note:** This step only defines the structure of the Silver table. Extraction, cleaning, validation, and loading of PullRequestEvent records from Bronze are performed in later steps.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.pull_request_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Event-specific fields
    pr_action         STRING,
    pr_number         BIGINT,
    pr_id             BIGINT,
    pr_created_at     TIMESTAMP,
    pr_closed_at      TIMESTAMP,
    pr_merged_at      TIMESTAMP,
    pr_merged         BOOLEAN,
    pr_commits        BIGINT,
    pr_additions      BIGINT,
    pr_deletions      BIGINT,
    pr_changed_files  BIGINT,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP
)
USING DELTA;